# Fuzze Trace Theory
### For evaluating semantic & conceptual quality

In [ ]:
import sys
sys.path.insert(0, "/media/am/AM/mi-lens/src")

from methods import (
    FuzzyTraceConfig,
    LensTokenMetrics,
    evaluate_token_metrics,
)

In [ ]:
import sys
import importlib
from pathlib import Path

import torch
import transformers

ROOT = Path("/media/am/AM/mi-lens")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "lenses" / "jacobian_lens"))
sys.path.insert(0, str(ROOT / "lenses" / "tuned_logit_lens"))

import jlens
from tuned_lens import TunedLens



In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
JLENS_PATH = ROOT / "tmp" / "jlens_out" / "ckpt.pt"
TUNED_LENS_PATH = ROOT / "tmp" / "tuned_lens_tinyllama_wikitext100_v2"

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
).cpu()

tok = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tok)
lens = jlens.JacobianLens.load(str(JLENS_PATH))
tuned_lens = TunedLens.from_model_and_pretrained(hf_model, str(TUNED_LENS_PATH))



In [ ]:
import pandas as pd
from dataclasses import asdict
from methods import FuzzyTraceConfig, evaluate_token_metrics

prompt = "Fact: The currency used in the country shaped like a boot is"
positions = [-2]  # or a small list like [-4, -3, -2, -1]
layers = lens.source_layers

jl_logits, final_logits, input_ids = lens.apply(
    model,
    prompt,
    layers=layers,
    positions=positions,
    use_jacobian=True,
)

ll_logits, _, _ = lens.apply(
    model,
    prompt,
    layers=layers,
    positions=positions,
    use_jacobian=False,
)

with torch.no_grad():
    outputs = hf_model(
        input_ids=input_ids.to(model.input_device),
        output_hidden_states=True,
        use_cache=False,
    )

hidden_states = outputs.hidden_states[:-1]
tuned_device = next(tuned_lens.parameters()).device

tuned_logits = {}
for layer in layers:
    tuned_idx = layer + 1
    hidden = hidden_states[tuned_idx][0]
    selected = hidden[list(positions)].to(tuned_device)
    tuned_logits[layer] = tuned_lens(selected, idx=tuned_idx).float().cpu()

token_ids = input_ids[0].tolist()
cfg = FuzzyTraceConfig(top_k=5)

rows = []
for j, pos in enumerate(positions):
    gold_token_id = token_ids[pos + 1]
    for layer in layers:
        jl = evaluate_token_metrics(
            jl_logits[layer][j],
            final_logits[j],
            gold_token_id,
            config=cfg,
        )
        ll = evaluate_token_metrics(
            ll_logits[layer][j],
            final_logits[j],
            gold_token_id,
            config=cfg,
        )
        tl = evaluate_token_metrics(
            tuned_logits[layer][j],
            final_logits[j],
            gold_token_id,
            config=cfg,
        )

        rows.append({
            "position": pos,
            "layer": layer,
            "lens": "jlens",
            **asdict(jl),
        })
        rows.append({
            "position": pos,
            "layer": layer,
            "lens": "logit_lens",
            **asdict(ll),
        })
        rows.append({
            "position": pos,
            "layer": layer,
            "lens": "tuned_lens",
            **asdict(tl),
        })

df = pd.DataFrame(rows)
df.head()



In [ ]:
df.groupby(["lens"])[["top1_exact", "topk_exact"]].mean()

In [ ]:
df.groupby(["lens", "layer"])[["gold_rank", "gold_prob", "topk_jaccard_vs_final"]].mean().reset_index()

#### Rule is:
- gold_rank: lower is better
- gold_prob: higher is better
- topk_jaccard_vs_final: higher is better
- js_vs_final / kl_vs_final / tv_vs_final: lower is better

In [ ]:
import plotly.express as px

plot_df = (
    df.groupby(["lens", "layer"], as_index=False)
    [["gold_rank", "gold_prob", "topk_jaccard_vs_final", "js_vs_final"]]
    .mean()
)

px.line(plot_df, x="layer", y="topk_jaccard_vs_final", color="lens", markers=True)

## Part 2

In [ ]:
import torch
import pandas as pd
from dataclasses import asdict

from methods import (
    LensEvalConfig,
    evaluate_surface_metrics,
    evaluate_residual_alignment,
)

prompt = "Fact: The currency used in the country shaped like a boot is"
positions = [-2]
layers = lens.source_layers
cfg = LensEvalConfig(top_k=5, char_ngram=3)

jl_logits, final_logits, input_ids = lens.apply(
    model,
    prompt,
    layers=layers,
    positions=positions,
    use_jacobian=True,
)

ll_logits, _, _ = lens.apply(
    model,
    prompt,
    layers=layers,
    positions=positions,
    use_jacobian=False,
)

with torch.no_grad():
    outputs = hf_model(
        input_ids=input_ids.to(model.input_device),
        output_hidden_states=True,
        use_cache=False,
    )

all_hidden = outputs.hidden_states
final_hidden = all_hidden[-1][0]
token_ids = input_ids[0].tolist()

tuned_device = next(tuned_lens.parameters()).device
tuned_logits = {}
for layer in layers:
    tuned_idx = layer + 1
    hidden = all_hidden[tuned_idx][0]
    selected = hidden[list(positions)].to(tuned_device)
    tuned_logits[layer] = tuned_lens(selected, idx=tuned_idx).float().cpu()

surface_rows = []
mech_rows = []

for j, pos in enumerate(positions):
    gold_token_id = token_ids[pos + 1]

    for layer in layers:
        source_hidden_cpu = all_hidden[layer + 1][0][pos].float().cpu()
        final_residual_cpu = final_hidden[pos].float().cpu()

        jl_surface = evaluate_surface_metrics(
            jl_logits[layer][j],
            final_logits[j],
            gold_token_id,
            tokenizer=tok,
            config=cfg,
        )
        ll_surface = evaluate_surface_metrics(
            ll_logits[layer][j],
            final_logits[j],
            gold_token_id,
            tokenizer=tok,
            config=cfg,
        )
        tl_surface = evaluate_surface_metrics(
            tuned_logits[layer][j],
            final_logits[j],
            gold_token_id,
            tokenizer=tok,
            config=cfg,
        )

        jl_residual = lens.transport(source_hidden_cpu.unsqueeze(0), layer)[0].float().cpu()
        ll_residual = source_hidden_cpu
        tl_residual = tuned_lens.transform_hidden(
            all_hidden[layer + 1][0][pos].to(tuned_device).unsqueeze(0),
            idx=layer + 1,
        )[0].float().cpu()

        jl_mech = evaluate_residual_alignment(jl_residual, final_residual_cpu)
        ll_mech = evaluate_residual_alignment(ll_residual, final_residual_cpu)
        tl_mech = evaluate_residual_alignment(tl_residual, final_residual_cpu)

        surface_rows.append({"position": pos, "layer": layer, "lens": "jlens", **asdict(jl_surface)})
        surface_rows.append({"position": pos, "layer": layer, "lens": "logit_lens", **asdict(ll_surface)})
        surface_rows.append({"position": pos, "layer": layer, "lens": "tuned_lens", **asdict(tl_surface)})

        mech_rows.append({"position": pos, "layer": layer, "lens": "jlens", **asdict(jl_mech)})
        mech_rows.append({"position": pos, "layer": layer, "lens": "logit_lens", **asdict(ll_mech)})
        mech_rows.append({"position": pos, "layer": layer, "lens": "tuned_lens", **asdict(tl_mech)})

surface_df = pd.DataFrame(surface_rows)
mech_df = pd.DataFrame(mech_rows)

surface_df.head(), mech_df.head()

In [ ]:
surface_df.groupby(["lens", "layer"], as_index=False)[
    [
        "top1_norm_matches_gold",
        "topk_contains_norm_gold",
        "best_topk_charngram_jaccard_gold",
        "best_topk_charngram_jaccard_final_top1",
    ]
].mean()

In [ ]:
mech_df.groupby(["lens", "layer"], as_index=False)[
    [
        "cosine_to_final",
        "relative_l2_to_final",
        "norm_ratio_to_final",
    ]
].mean()

Interpretation:
- higher best_topk_charngram_jaccard_gold is better for softer surface/gist overlap
- higher cosine_to_final is better for mechanistic alignment
- lower relative_l2_to_final is better for mechanistic alignment

In [ ]:
import plotly.express as px

surface_plot_df = (
    surface_df.groupby(["lens", "layer"], as_index=False)[
        [
            "top1_norm_matches_gold",
            "topk_contains_norm_gold",
            "best_topk_charngram_jaccard_gold",
            "best_topk_charngram_jaccard_final_top1",
        ]
    ]
    .mean()
)

fig = px.line(
    surface_plot_df,
    x="layer",
    y="best_topk_charngram_jaccard_gold",
    color="lens",
    markers=True,
    title="Surface/Gist Proxy vs Layer",
)
fig.update_layout(
    xaxis_title="Layer",
    yaxis_title="Best top-k char-ngram Jaccard vs gold",
)
fig.show()

In [ ]:
import plotly.express as px

mech_plot_df = (
    mech_df.groupby(["lens", "layer"], as_index=False)[
        [
            "cosine_to_final",
            "relative_l2_to_final",
            "norm_ratio_to_final",
        ]
    ]
    .mean()
)

fig = px.line(
    mech_plot_df,
    x="layer",
    y="cosine_to_final",
    color="lens",
    markers=True,
    title="Mechanistic Faithfulness vs Layer",
)
fig.update_layout(
    xaxis_title="Layer",
    yaxis_title="Cosine to actual final residual",
)
fig.show()

In [ ]:
px.line(
    surface_plot_df,
    x="layer",
    y="topk_contains_norm_gold",
    color="lens",
    markers=True,
    title="Normalized Gold Token Containment vs Layer",
).show()

In [ ]:
px.line(
    mech_plot_df,
    x="layer",
    y="relative_l2_to_final",
    color="lens",
    markers=True,
    title="Relative L2 to Final Residual vs Layer",
).show()